# 🛡️ AI-Powered Network Intrusion Detection System — Demo

[![GitHub](https://img.shields.io/badge/GitHub-ai--network--threat--detection-black?logo=github)](https://github.com/dheerajramasahayam/ai-network-threat-detection)

This notebook walks through the complete AI-NIDS pipeline end-to-end:

1. **Install dependencies**
2. **Load the pre-trained model**
3. **Classify network flows** (benign vs attack)
4. **SHAP explanations** — why did the model flag this flow?
5. **Zero-day anomaly detection**
6. **Adversarial robustness test**
7. **Per-attack-class analysis**
8. **Inference latency benchmark**


## 1. Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/dheerajramasahayam/ai-network-threat-detection.git
%cd ai-network-threat-detection
!pip install -q -r requirements.txt
print('✅ Setup complete')

In [ ]:
import sys, os
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from src.model import IntrusionDetectionModel
from src.detection import ThreatDetector
from src.preprocessing import FEATURE_COLUMNS

print(f'Feature schema: {len(FEATURE_COLUMNS)} canonical features')
print('Imports OK ✅')

## 2. Load Pre-Trained Models

We ship two models with the repo:
- `random_forest.joblib` — 99.86% accuracy, 277K flows/sec
- `xgboost.joblib` — 99.85% accuracy, 1.54M flows/sec


In [ ]:
import glob

model_files = [f for f in glob.glob('models/*.joblib') if 'zero_day' not in f]
print('Available models:', model_files)

rf_model = IntrusionDetectionModel.load('models/random_forest.joblib')
xgb_model = IntrusionDetectionModel.load('models/xgboost.joblib')
print('✅ Models loaded')

## 3. Classify Network Flows

Classify synthetic flow vectors representing:
- A normal HTTPS web browsing session
- A DDoS volumetric attack
- A slow-rate port scanner


In [ ]:
# Create 3 representative synthetic flow vectors (41 features, standardized)
rng = np.random.default_rng(42)

flows = {
    'Normal HTTPS session': rng.normal(loc=0.0, scale=0.5, size=41),   # Small, normal
    'DDoS flood':           rng.normal(loc=5.0, scale=2.0, size=41),   # High-volume, large packets
    'Port scanner':         rng.normal(loc=-2.0, scale=1.0, size=41),  # Many short connections
}

X = np.array(list(flows.values()))

print(f'{'Flow':<30} {'RF Prob':>10} {'RF Label':>10} {'XGB Prob':>10} {'XGB Label':>10}')
print('-' * 75)
for name, x in flows.items():
    x2d = x.reshape(1, -1)
    rf_prob  = rf_model.predict_proba(x2d)[0]
    xgb_prob = xgb_model.predict_proba(x2d)[0]
    rf_label  = 'ATTACK' if rf_prob >= 0.5 else 'BENIGN'
    xgb_label = 'ATTACK' if xgb_prob >= 0.5 else 'BENIGN'
    print(f'{name:<30} {rf_prob:>10.4f} {rf_label:>10} {xgb_prob:>10.4f} {xgb_label:>10}')

## 4. SHAP Explanations

For each prediction: *which features contributed most to the model's decision?*


In [ ]:
try:
    import shap

    # Use TreeExplainer for RF (fast, exact)
    explainer = shap.TreeExplainer(rf_model.model)
    shap_vals = explainer.shap_values(X)  # shape: (n_samples, n_features, n_classes)

    # Plot SHAP for the DDoS flow (index 1)
    i = 1  # DDoS flood
    vals = shap_vals[1][i] if isinstance(shap_vals, list) else shap_vals[i, :, 1]
    top = np.argsort(np.abs(vals))[::-1][:10]

    plt.figure(figsize=(9, 4))
    colors = ['#dc2626' if v > 0 else '#2563eb' for v in vals[top]]
    plt.barh([FEATURE_COLUMNS[j] for j in top[::-1]], vals[top[::-1]], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('SHAP Feature Importances — DDoS Flow\nRed = pushes toward ATTACK, Blue = toward BENIGN',
              fontsize=12, fontweight='bold')
    plt.xlabel('SHAP value')
    plt.tight_layout()
    plt.show()
    print('SHAP explanation ✅')
except Exception as e:
    print(f'SHAP not available in this environment: {e}')
    print('Run locally with: pip install shap')

## 5. Zero-Day Anomaly Detection

Unsupervised ensemble (IsolationForest + OCSVM + LOF) detects anomalies with no prior knowledge of attack signatures.


In [ ]:
from src.anomaly import ZeroDayDetector

# Train detector on 'normal' traffic
X_normal = rng.normal(loc=0.0, scale=0.5, size=(2000, 41))
detector = ZeroDayDetector()
detector.fit(X_normal)

# Score test flows
test_flows = np.vstack([
    rng.normal(0.0, 0.5, (5, 41)),   # normal
    rng.normal(5.0, 2.0, (5, 41)),   # anomalous
])
scores = detector.score_samples(test_flows)

print('Anomaly scores (lower = more anomalous):')
labels_true = ['Normal'] * 5 + ['Anomaly'] * 5
for label, score in zip(labels_true, scores):
    flag = '🔴 ANOMALY' if score < 0 else '🟢 Normal'
    print(f'  {label:<10} score={score:>8.4f}  →  {flag}')

## 6. Adversarial Robustness

How much noise does it take to fool the model?


In [ ]:
# Simulate adversarial perturbations of attack flows
X_attack = X[1:2]  # DDoS flow
epsilons = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]

print(f'Original DDoS prob: {rf_model.predict_proba(X_attack)[0]:.4f}')
print()
print(f'{"Epsilon":<10} {"After noise prob":<20} {"Label"}')
print('-' * 40)

for eps in epsilons:
    noise = rng.uniform(-eps, eps, X_attack.shape)
    X_adv = X_attack + noise
    prob = rf_model.predict_proba(X_adv)[0]
    label = '🔴 ATTACK' if prob >= 0.5 else '✅ EVADED'
    print(f'{eps:<10.3f} {prob:<20.4f} {label}')

## 7. Inference Latency Benchmark

How many flows/second can the models classify?


In [ ]:
import time

X_pool = rng.normal(size=(50_000, 41))
batch_sizes = [1, 10, 100, 1_000, 10_000]

print(f'{"Model":<20} {"Batch":<10} {"ms total":<12} {"flows/sec"}')
print('-' * 55)

for model_name, model in [('RandomForest', rf_model), ('XGBoost', xgb_model)]:
    for bs in batch_sizes:
        # Warm up
        model.predict_proba(X_pool[:bs])

        t0 = time.perf_counter()
        for _ in range(20):
            model.predict_proba(X_pool[:bs])
        elapsed_ms = (time.perf_counter() - t0) / 20 * 1000
        fps = bs / (elapsed_ms / 1000)
        print(f'{model_name:<20} {bs:<10,} {elapsed_ms:<12.2f} {fps:,.0f}')

## 8. Load Results from Pre-Run Benchmarks

View the full benchmark reports that were pre-computed on the full 3.87M dataset.


In [ ]:
from IPython.display import Markdown, Image, display
import os

for report in [
    'results/accuracy_report_combined.md',
    'results/attack_report.md',
    'results/adversarial_report.md',
    'results/latency_benchmark.md',
]:
    if os.path.exists(report):
        with open(report) as f:
            display(Markdown(f'---\n{f.read()}'))

# Show visualizations
for img in [
    'results/cross_dataset_heatmap.png',
    'results/attack_precision_recall.png',
    'results/adversarial_robustness.png',
    'results/latency_throughput.png',
]:
    if os.path.exists(img):
        display(Image(img, width=800))

---

## Next Steps

- 📦 **Download multi-datasets**: `python src/dataset_downloader.py` (requires `KAGGLE_API_TOKEN`)
- 🔁 **Retrain on combined.csv**: `python src/training.py --dataset dataset/combined.csv --label-col label`
- 🌐 **Start the API**: `uvicorn src.api:app --reload`
- ⭐ **Star the repo**: [github.com/dheerajramasahayam/ai-network-threat-detection](https://github.com/dheerajramasahayam/ai-network-threat-detection)
